# 19 - SimFT: personalized aggregation weighted by function-space similarity (+ a final local fine-tune)

Pre-registered in `docs/2026-10-28-simft-preregistration.md` (read it first; it also lists the evidence behind the design).

**The method.** Each round, after local training, every hospital's model is scored on every hospital's validation
half A (the receiving hospital would do this itself; only the loss numbers matter). Hospital *i*'s next model is a
weighted average of all hospitals' models: it keeps at least 25% of its own, and the rest is a softmax over how much
*better* each neighbour's model is on *its* data (temperature tau). After the last round every hospital fine-tunes its
own model for 8 epochs. Validation half B (never used for the weights) selects round, cutoff and epoch; test is read once.
The ablation `SimFT-flat` keeps the same structure but weights neighbours equally, so any gain over it comes from the similarity.

**Two settings, one partition per seed (5 seeds):**
1. **conflict** - `antibiogram_conflict(n_clients=5)` (hospitals disagree about the same bug-drug pairs). Arms: FedAvg (+ local-only +
   fine-tune), SimFT at tau = 0.5 and 1.0 (best chosen on validation half B), SimFT-flat.
2. **ward10** - ten ward-mix hospitals keeping 10% of their train/validation patients (the notebook-18 setting).
   Arms: SimFT tau = 0.5 / 1.0 and SimFT-flat only; local-only, FedAvg and FedAvg + fine-tune are **read from notebook 18**
   (`results/amr_regimes_all.json`, key `scarcity_f0.1`; same seeds, partitions and subsample). Put that file in `results/` first.

Primary metric: the **unweighted mean over hospitals** of test macro-F1, every hospital at its own validation-chosen cutoff.
**Win rule:** SimFT beats FedAvg + fine-tune, local-only and SimFT-flat, each with a paired 95% interval above zero.
Budget: rounds 8, local epochs 4, local-only epochs 32, hidden 128 (same as notebooks 16-18). Resume-safe seed by seed;
results in `results/amr_simft_<setting>_<TAG>.json` (mirrored to Drive on Colab). Run on the same kind of machine as notebook 18
(Colab T4) if you can: the ward10 baselines came from there.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/federated-baselines'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this is the final-protocol branch (modules below must exist)
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import final_protocol as P, analysis as A, selection, simft, simft_runs as SR, regimes
from amr_fed.topology import personalized_weights, train_frame
print('imports OK | v3 temperatures:', P.T_GRID, '| fine-tune epochs:', P.FT_EPOCHS)

In [ ]:
# 3) Load cohort + smoke: both partitions valid
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import antibiogram_conflict, dirichlet_ward_mixture, partition_summary
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
a = antibiogram_conflict(df, n_clients=5)
g, niid = partition_summary(df, a); print(g)
assert a.index.is_unique and len(a) == df[PK].nunique(), 'conflict split not 1:1'
assert (g['n_tests'] / g['n_tests'].sum() >= 0.03).all(), 'hospital below the 3% size floor'
assert g['resist_rate'].max() - g['resist_rate'].min() >= 0.15, 'resistance-rate spread too small'
for s in SEEDS:
    w = dirichlet_ward_mixture(df, n_clients=10, alpha=0.5, seed=s)
    print(f'ward seed {s}: patients (largest first): {w.value_counts().sort_values(ascending=False).tolist()}')
print('SMOKE PASS')

In [ ]:
# 4) SETTING 1 - conflict split (resume-safe). Arms: FedAvg(+local-only,+fine-tune), SimFT tau=0.5, tau=1.0, SimFT-flat.
import os, ray, json
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed') and not any(x in m for x in ('final_protocol', 'simft_runs', 'regimes'))]:
    del _s.modules[_m]
from amr_fed.federated import final_protocol as P, analysis as A, simft_runs as SR
from amr_fed.partition import antibiogram_conflict, dirichlet_ward_mixture
TAG = 'all'
KW_C = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: antibiogram_conflict(d, n_clients=5, seed=s),
            n_clients=5, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RES_C = SRC.parent / 'results' / f'amr_simft_conflict_{TAG}.json'
RES_C.parent.mkdir(exist_ok=True)
MIR_C = Path(f'/content/drive/MyDrive/amr_simft_conflict_{TAG}.json') if ON_COLAB else None
src = RES_C if RES_C.exists() else (MIR_C if MIR_C and MIR_C.exists() else None)
res_c = json.load(open(src)) if src else {}
res_c = SR.run_setting(res_c, 'conflict', kw=KW_C, checkpoint=lambda r: P.save_json(r, RES_C, MIR_C), ray_shutdown=ray.shutdown)
P.save_json(res_c, RES_C, MIR_C)
print('conflict done:', [k for k in res_c if not k.startswith('_')])

In [ ]:
# 5) SETTING 2 - ward10 at 10% data. Only the SimFT arms are trained; baselines are read from notebook 18's results file.
import glob
cand = [SRC.parent / 'results' / 'amr_regimes_all.json'] + ([Path('/content/drive/MyDrive/amr_regimes_all.json')] if ON_COLAB else [])
base_file = next((p for p in cand if p.exists()), None)
assert base_file, 'notebook 18 results (amr_regimes_all.json) not found - put it in results/ (or Drive) and re-run this cell'
base = json.load(open(base_file))['scarcity_f0.1']
print('baselines (local-only, FedAvg, FedAvg+fine-tune) from', base_file, '| seeds', base['seeds'])
KW_W = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: dirichlet_ward_mixture(d, n_clients=10, alpha=0.5, seed=s),
            n_clients=10, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RES_W = SRC.parent / 'results' / f'amr_simft_ward10_{TAG}.json'
MIR_W = Path(f'/content/drive/MyDrive/amr_simft_ward10_{TAG}.json') if ON_COLAB else None
src = RES_W if RES_W.exists() else (MIR_W if MIR_W and MIR_W.exists() else None)
res_w = json.load(open(src)) if src else {}
res_w['fedavg'] = base                       # local-only / FedAvg / FedAvg+fine-tune from notebook 18 (same seeds, partitions, 10% subsample)
res_w = SR.run_setting(res_w, 'ward10', kw=KW_W, checkpoint=lambda r: P.save_json(r, RES_W, MIR_W), ray_shutdown=ray.shutdown)
P.save_json(res_w, RES_W, MIR_W)
print('ward10 done:', [k for k in res_w if not k.startswith('_')])

In [ ]:
# 6) Results - conflict split
print(A.simft_report(res_c)); print(); print(A.simft_weights_report(res_c))

In [ ]:
# 7) Results - ward10 at 10% data
print(A.simft_report(res_w)); print(); print(A.simft_weights_report(res_w))

In [ ]:
# 8) Compute / provenance
import numpy as np
for name, res_ in (('conflict', res_c), ('ward10', res_w)):
    for k, v in res_.items():
        if k.startswith('_') or not isinstance(v, dict) or 'runs' not in v or 'wall_seconds' not in v['runs'][0]:
            continue
        print(name, k, 'min per seed (mean):', round(np.mean([r['wall_seconds'] for r in v['runs']]) / 60, 1), '| gpu:', v['runs'][0]['provenance'].get('gpu'), '| git:', v['runs'][0]['provenance'].get('git'))